# 第3课 训练AI大脑

适合对象：七年级同学（已经完成第2课预处理）

前置知识：
- 知道“输入 -> 模型 -> 输出”基本流程
- 能看懂 Python 函数和循环

学习目标：
- 学会加载训练数据
- 学会搭建一个简单神经网络（CNN）
- 学会训练模型并观察准确率变化
- 学会保存模型文件，供下一课预测使用


## 课程流程

1. 导入深度学习库并设定随机种子
2. 加载数据集（优先本地，找不到就用示例数据）
3. 搭建神经网络
4. 训练模型
5. 保存模型
6. 课堂练习


In [ ]:
# 第1步：导入库
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, random_split
    from torchvision import datasets, transforms
except Exception as e:
    raise ImportError(
        '本课需要 PyTorch 与 torchvision。请先安装：pip install torch torchvision'
    ) from e

# 固定随机种子，让同样代码多次运行时结果更稳定
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 如果电脑有 GPU，会更快；没有也没关系，CPU 也能跑
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('当前计算设备:', device)


## Step 1 加载数据

我们优先读取你自己的张江建筑数据集（`ImageFolder` 结构）。

如果本地还没有准备好数据，Notebook 会自动切换到 `FakeData` 演示流程，
这样你依然可以完整学习“加载 -> 训练 -> 保存”。


In [ ]:
# 第2步：定义数据预处理方式
# 说明：Resize 把图片统一到 64x64；ToTensor 把像素转成张量；Normalize 做标准化
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# 依次尝试多个路径，便于不同同学的目录结构
data_candidates = [
    Path('../data/zhangjiang_dataset'),
    Path('data/zhangjiang_dataset'),
    Path('../dataset/zhangjiang_dataset'),
]

data_dir = next((p for p in data_candidates if p.exists()), None)

if data_dir is not None:
    full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    class_names = full_dataset.classes
    print(f'使用本地数据集: {data_dir}')
else:
    # 没有本地数据时，用 FakeData 模拟 3 类建筑
    class_names = ['双子塔', '上海科技馆', '张江药谷']
    full_dataset = datasets.FakeData(
        size=900,
        image_size=(3, 64, 64),
        num_classes=len(class_names),
        transform=transform,
    )
    print('未找到本地数据集，当前使用 FakeData 演示。')

# 按 8:2 划分训练集与验证集
train_size = int(len(full_dataset) * 0.8)
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# DataLoader 负责“分批喂数据”给模型
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print('总样本数:', len(full_dataset))
print('训练样本数:', len(train_dataset))
print('验证样本数:', len(val_dataset))
print('类别名称:', class_names)

# 抽取一个批次可视化
imgs, labels = next(iter(train_loader))
plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    # 训练前做了 Normalize，这里把图像临时反归一化再显示
    show_img = (imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    plt.imshow(show_img)
    plt.title(class_names[int(labels[i]) % len(class_names)], fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


## Step 2 搭建神经网络

我们搭建一个“小型 CNN”：
- 卷积层：提取边缘、纹理等视觉特征
- 池化层：压缩信息，减少计算量
- 全连接层：根据特征做分类判断


In [ ]:
# 第3步：定义一个简单的卷积神经网络
class SmallCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        # 特征提取部分
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8
        )
        # 分类头
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SmallCNN(num_classes=len(class_names)).to(device)
print(model)

# 统计参数量：参数越多，模型表达能力通常越强，但训练也更慢
total_params = sum(p.numel() for p in model.parameters())
print(f'模型总参数量: {total_params:,}')


## Step 3 训练模型

训练的核心循环：
1. 前向传播：模型给出预测
2. 计算损失：预测与真实标签差多少
3. 反向传播：把“错误信息”传回去
4. 更新参数：让模型下一次更接近正确答案


In [ ]:
# 第4步：训练与验证
loss_fn = nn.CrossEntropyLoss()          # 多分类常用损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5
history = {'train_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    # ---- 训练阶段 ----
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()             # 清空上一步梯度
        logits = model(x_batch)           # 前向传播
        loss = loss_fn(logits, y_batch)   # 计算损失
        loss.backward()                   # 反向传播
        optimizer.step()                  # 更新参数

        running_loss += loss.item() * x_batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # ---- 验证阶段 ----
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            pred = logits.argmax(dim=1)
            correct += (pred == y_batch).sum().item()
            total += y_batch.size(0)

    val_acc = correct / total if total > 0 else 0.0
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)

    print(f'第 {epoch+1:02d} 轮 | 训练损失: {train_loss:.4f} | 验证准确率: {val_acc:.2%}')

# 画出训练曲线
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], marker='o')
plt.title('训练损失变化')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], marker='o', color='green')
plt.title('验证准确率变化')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()


## Step 4 保存模型

训练结束后要保存模型，不然下次还得重新训练。

我们会保存：
- 模型参数 `model_state_dict`
- 类别名称 `class_names`
- 输入尺寸与标准化参数（方便预测时保持一致）


In [ ]:
# 第5步：保存模型到 ../models/lesson03_cnn.pth
model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'lesson03_cnn.pth'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'input_size': [3, 64, 64],
    'normalize': {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]},
    'num_epochs': num_epochs,
    'seed': SEED,
}
torch.save(checkpoint, model_path)

print('模型已保存:', model_path.resolve())


## 课堂练习

1. 把 `num_epochs` 改成 `10`，观察准确率是否提高。
2. 把 `batch_size` 改成 `16` 或 `64`，比较训练速度和稳定性。
3. 思考：为什么训练集准确率高，不代表真实效果一定好？

常见坑：
- 训练时忘记 `model.train()`，验证时忘记 `model.eval()`。
- 预测时没有和训练时使用同样的标准化参数。


In [ ]:
# 练习答案脚手架：写一个小函数，计算一批图片的预测准确率
def batch_accuracy(model, batch_x, batch_y, device='cpu'):
    model.eval()
    with torch.no_grad():
        logits = model(batch_x.to(device))
        pred = logits.argmax(dim=1)
        acc = (pred == batch_y.to(device)).float().mean().item()
    return acc

sample_x, sample_y = next(iter(val_loader))
acc = batch_accuracy(model, sample_x, sample_y, device=device)
print(f'一个验证批次的准确率: {acc:.2%}')
